# Dimensionsreduktion

**Dimensionsreduktion**: Wie projizieren wir hochdimensionale Daten auf wenige Dimensionen, sodass möglichst viel der "interessanten" Struktur erhalten bleibt?

### Themen in diesem Notebook
1. Die Hauptkomponentenanalyse (PCA)
2. Warum Standardisierung vor PCA wichtig ist
3. Den Unterschied zwischen linearen und nichtlinearen Verfahren
4. Vor- und Nachteile der Verfahren

> **Hinweis zu den Daten:** Hier verwenden wir die folgenden Dateien: `penguins.csv` und `digits.csv.gz`. Bitte laden Sie sie herunter und stellen Sie sicher, dass die Dateien im selben Ordner wie dieses Notebook liegen.
>

## Benötigte Module und Reproduzierbarkeit

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import make_circles

RANDOM_STATE = 19751979
generator = np.random.default_rng(RANDOM_STATE)

## Teil 1 — Intuition mit zwei Merkmalen

Fürs erste Beispiel werden wir den uns schon bekannten Palmer-Penguins-Datensatz laden.

In [ ]:
def load_penguins(file_csv: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Lädt eine vereinfachte Version des "Palmer Penguins"-Datensatzes.
    :param file_csv: Pfad zur CSV-Datei mit den folgenden 5 Spalten:
                     species, bill length, bill depth, flipper length, body mass.
    :return: Tupel `(x, y, species)`, wobei:
             `x` einen Numpy-Array der Form `(n, 4)` ist, der die Merkmalswerte beinhaltet;
             `y` einen Numpy-Array der Form `(n,)` ist, der die Kategorieindizes beinhaltet;
             `species` eine Liste mit allen Pinguinarten ist.
    """
    expected_header = ['species', 'bill length', 'bill depth', 'flipper length', 'body mass']
    delimiter = ','

    # Alle Zeilen der Datei parsen
    x = []  # Merkmalswerte
    penguins = []  # alle Pinguinarten
    with open(file_csv, 'r') as file_handle:
        for index_line, line in enumerate(file_handle):
            line = line.strip()
            if index_line == 0:
                if line != delimiter.join(expected_header):
                    raise ValueError('Unerwartete Header-Zeile')
            elif line != '':
                values = line.split(delimiter)
                if len(values) != len(expected_header):
                    raise ValueError(f'Unerwartete Spalenzahl in Zeile {index_line + 1}')
                x.append([float(x) for x in values[1:]])
                penguins.append(values[0])

    # Artennamen in Indizes umwandeln
    species = list(set(penguins))
    species.sort()
    y = [species.index(current_name) for current_name in penguins]

    # Listen mit Zahlen zu Numpy-Arrays konvertieren
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.uint16)
    return x, y, species


# Bitte Pfad zur CSV-Datei anpassen
file_dataset = 'penguins.csv'
x_penguins, y_penguins, species = load_penguins(file_dataset)
feature_names = [
    'Schnabellänge (mm)', 'Schnabeltiefe (mm)',
    'Flossenlänge (mm)', 'Körpermasse (g)']

# Info über den Datensatz:
print(f'Dimensionalität von x: {x_penguins.shape}')
print(f'Dimensionalität von y: {y_penguins.shape}')
print(f'          Pinguinarte: {species}')

Hochdimensionale Daten kann man nicht direkt zeichnen. Deshalb beginnen wir mit den folgenden zwei Merkmalen:

- `Schnabellänge (mm)` — Schnabellänge in Millimetern (ca. 30–60)
- `Körpermasse (g)` — Körpermasse in Gramm (ca. 2700–6300)

Achten Sie schon jetzt auf die sehr unterschiedlichen Wertebereiche.

In [ ]:
# Zwei Merkmale zu einer Matrix (n_samples, 2) zusammensetzen
x_raw = x_penguins[:, [0, 3]]

print(f'            Form von x_raw: {x_raw.shape}')
print(f'Wertebereich Schnabellänge: [{x_raw[:, 0].min()}, {x_raw[:, 0].max()}]')
print(f'  Wertebereich Körpermasse: [{x_raw[:, 1].min()}, {x_raw[:, 1].max()}]')

Wir können diese Merkmale in der Ebene durch ein Streudiagramm darstellen.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 4.5), dpi=100, sharey='all')
axes[0].scatter(x_raw[:, 0], x_raw[:, 1], s=15, alpha=0.6)
axes[0].set(xlabel=feature_names[0], ylabel=feature_names[3])
axes[1].scatter(x_raw[:, 0], x_raw[:, 1], s=15, alpha=0.6)
axes[1].set(aspect='equal', xlabel=feature_names[0])
figure.tight_layout()
plt.show(figure)
del figure, axes

### Wie funktioniert PCA? (Die Mathematik dahinter)

PCA sucht neue, zueinander **orthogonale** Achsen (die *Hauptkomponenten*), entlang derer die Daten
die **größte Varianz** haben. Die erste Hauptkomponente ist also die Richtung, in der die Punkte am
weitesten gestreut sind — anschaulich die Richtung, in der unser "Schatten" am meisten Information behält.

**Schritt 1 — Zentrieren.** Vom Mittelwert $\bar{\mathbf{x}}$ abziehen:
$$\tilde{\mathbf{X}} = \mathbf{X} - \bar{\mathbf{x}}$$

**Schritt 2 — Kovarianzmatrix** der zentrierten Daten ($n$ = Anzahl Datenpunkte):
$$\mathbf{C} = \frac{1}{n-1}\,\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}$$

**Schritt 3 — Eigenzerlegung.** Die Eigenvektoren $\mathbf{v}_i$ von $\mathbf{C}$ sind die Hauptkomponenten,
die zugehörigen Eigenwerte $\lambda_i$ geben die Varianz entlang dieser Richtung an:
$$\mathbf{C}\,\mathbf{v}_i = \lambda_i \mathbf{v}_i$$

**Schritt 4 — Projektion** auf die $k$ wichtigsten Komponenten (Eigenvektoren als Spalten in $\mathbf{V}_k$):
$$\mathbf{Z} = \tilde{\mathbf{X}}\,\mathbf{V}_k$$

Der **Anteil der erklärten Varianz** einer Komponente ist $\dfrac{\lambda_i}{\sum_j \lambda_j}$.

`sklearn` erledigt diese Schritte intern (numerisch stabil über die Singulärwertzerlegung) für uns.

### Warum Standardisierung?

PCA maximiert Varianz. Hat ein Merkmal allein wegen seiner Einheit viel größere Zahlenwerte (Gramm gegenüber Millimeter!), dominiert es die Varianz — und damit die erste Hauptkomponente — obwohl es nicht "wichtiger" sein muss. Mit **Standardisierung** bekommt jedes Merkmal
Mittelwert 0 und Standardabweichung 1:
$$x' = \frac{x - \mu}{\sigma}$$
Damit zählt jedes Merkmal gleichberechtigt.

### Übung

Der folgende Code-Ausschnitt berechnet die PCA auf den Rohdaten, indem ein [`PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)-Objekt erstellt und dessen Methode `fit` aufgerufen wird.

Berechnen Sie zusätlich die Hauptkomponenten **für die standardisierten Daten** und speichern Sie das entsprechende `PCA`-Objekt als Variable namens `pca_std`. Sie können die Spalten einer Matrix mit einem [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) standardisieren.

In [ ]:
# PCA auf den Rohdaten
pca_raw = PCA(n_components=2)
pca_raw.fit(x_raw)

# Standardisieren und PCA auf den standardisierten Daten
scaler = StandardScaler()
x_std = scaler.fit_transform(x_raw)
pca_std = PCA(n_components=2)
pca_std.fit(x_std)

print(f'           Erste Hauptkomponente (roh): {np.round(pca_raw.components_[0], 3)}')
print(f'Erste Hauptkomponente (standardisiert): {np.round(pca_std.components_[0], 3)}')
print(f'                Erklärte Varianz (roh): {np.round(pca_raw.explained_variance_ratio_, 5)}')
print(f'     Erklärte Varianz (standardisiert): {np.round(pca_std.explained_variance_ratio_, 3)}')

Wir können die berechneten Komponenten bei den Ansätzen visualisieren.

In [ ]:
def plot_pca_axes(ax, x: np.ndarray, pca: PCA):
    """
    Zeichnet die Hauptkomponenten als Pfeile in ein Streudiagramm.
    :param ax: `Axes`-Objekt für die Erstellung des Diagramms.
    :param x: zu visualisierender Datensatz.
    :param pca: auf `x` angepasstes `PCA`-Objekt.
    """
    ax.scatter(x[:, 0], x[:, 1], s=8, alpha=0.5)
    x_mean = x.mean(axis=0)
    # Pfeillaenge proportional zur Standardabweichung entlang der Komponente
    style = {'arrowstyle': '->', 'color': 'red', 'lw': 2}
    for variance, vector in zip(pca.explained_variance_, pca.components_):
        arrow = vector * 2.5 * np.sqrt(variance)
        ax.annotate('', xy=x_mean + arrow, xytext=x_mean, arrowprops=style)


figure, axes = plt.subplots(1, 2, figsize=(10, 4.5), dpi=100)
plot_pca_axes(axes[0], x_raw, pca_raw)
axes[0].set(xlabel=feature_names[0], ylabel=feature_names[3])
plot_pca_axes(axes[1], x_std, pca_std)
axes[1].set(xlabel='Schnabellänge (standardisiert)', ylabel='Körpermasse (standardisiert)')
plt.show()

### Beobachtung und Diskussion

Vergleichen Sie die roten Pfeile:

- **Ohne Standardisierung** zeigt die erste Hauptkomponente fast senkrecht nach oben — also praktisch nur entlang `Körpermasse (g)`. Sie "erklärt" nahezu 100 % der Varianz. Das liegt nicht daran, dass die Körpermasse informativer wäre, sondern nur daran, dass Gramm-Werte numerisch viel größer sind als Millimeter.
- **Mit Standardisierung** mischt die erste Hauptkomponente beide Merkmale (etwa $[0{,}71,\ 0{,}71]$) und folgt der tatsächlichen, schräg verlaufenden Korrelationsstruktur der Wolke.

**Merksatz:** Wenn Merkmale unterschiedliche Einheiten oder Skalen haben, sollte man vor der PCA standardisieren.

## Teil 2 — PCA auf hochdimensionalen Daten: Handgeschriebene Ziffern

Hier wenden wir PCA auf den [NIST-Datensatz]((https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits)) an. Er ist ein handgeschriebener Zifferndatensatz mit niedriger Auflösung. Der Datensatz beinhaltet 3823 Graustufenbilder von Ziffern. Jedes Bild hat die Abmessungen 8 x 8 Pixel, sowie ein Label - die gezeigte Ziffer 0 bis 9.

### Datensatz laden

In [ ]:
dataset = np.loadtxt('digits.csv.gz', delimiter=',', dtype=np.uint16)
x_digits = dataset[:, :-1]  # Pixel
y_digits = dataset[:, -1]  # Label
del dataset

print(f'    Form der Merkmalsmatrix: {x_digits.shape}')
print(f'Datentyp der Merkmalsmatrix: {x_digits.dtype}')
print(f'             Anzahl Klassen: {len(np.unique(y_digits))}')
print(f'          Pixelintensitäten: [{x_digits.min()}, {x_digits.max()}]')

Visualisieren wir 3 zufällige Bilder aus dem Datensatz.

In [ ]:
def plot_random_digits(generator, image_count: int, x: np.ndarray, y: np.ndarray) -> plt.Figure:
    """
    Visualisiert zufällig ausgewählte Bilder aus dem NIST-Datensatz.
    
    :param generator: `Generator`-Objekt zur Auswahl zufälliger Bilder.
    :param image_count: Anzahl an Bilder zur Auswahl.
    :param x: Mermalsmatrix als Array der Form `(N, 64)`.
    :param y: Labels der Bilder als Integer-Array der Form `(N,)`.
    :return: das neu erstellte Diagramm als `Figure`-Objekt.
    """
    image_indices = generator.choice(y.size, size=image_count, replace=False)
    figure, axes = plt.subplots(1, image_count, figsize=(2.3 * image_count, 2.6), dpi=100)
    for i, ax in zip(image_indices, axes):
        ax.imshow(x[i, :].reshape(8, 8), cmap='gray_r')
        ax.set_title(str(y[i]))
        ax.axis('off')
    return figure


figure = plot_random_digits(generator, 3, x_digits, y_digits)
plt.show(figure)

**Kurze Anmerkung zur Standardisierung hier:** Bei den Penguins mussten wir standardisieren, weil die Merkmale völlig verschiedene Einheiten hatten. Bei den Ziffern liegen alle 64 Pixel auf derselben Skala (0–16). Eine Standardisierung würde hier sogar fast konstante Randpixel künstlich aufblähen. Deshalb standardisieren wir die Bilder **nicht** — wir zentrieren nur, und das macht `PCA` automatisch.

### Übung: Wie viele Komponenten brauchen wir?

1. Passen Sie ein `PCA`-Objekt namens `pca_full` ohne Begrenzung der Komponentenanzahl an `x_digits` an.
2. Lesen Sie den Anteil der erklärten Varianz aus dem Attribut `explained_variance_ratio_` in die Variable `explained`.

Danach berechnen wir die **kumulierte** erklärte Varianz mit [`np.cumsum`](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html).

In [ ]:
pca_full = PCA()
pca_full.fit(x_digits)
explained = pca_full.explained_variance_ratio_

print(f'Erste 5 Komponenten erklären je: {np.round(explained[:5], 3)}')
print(f'   Kumuliert nach 5 Komponenten: {np.round(np.cumsum(explained[:5]), 3)}')

Schauen wir wie sich die erklärbare Varianz der Hauptkomponenten ändert. Das Liniendiagramm, das die erklärte Varianz für jede Komponente zeigt, wird [Scree-Plot](https://de.wikipedia.org/wiki/Scree-Test) genannt.

Wie viele Komponenten brauchen wir, um 90 % der Varianz zu behalten?

In [ ]:
# Scree-Plot (links) und kumulierte erklaerte Varianz (rechts)
figure, axes = plt.subplots(1, 2, figsize=(11, 3.8), dpi=100)
xs = np.arange(explained.size) + 1
axes[0].plot(xs, explained, marker='o', ms=3)
axes[0].set(axisbelow=True, xlim=[xs[0] - 0.5, xs[-1] + 0.5])
axes[0].set(xlabel='Komponente', ylabel='Erklärte Varianz')
axes[0].grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
axes[1].plot(xs, np.cumsum(explained), marker='o', ms=3)
axes[1].axhline(0.90, color='red', ls='--', label='90 %')
axes[1].set(axisbelow=True, xlim=[xs[0] - 0.5, xs[-1] + 0.5])
axes[1].set(xlabel='Anzahl Komponenten', ylabel='Kumulierte erklärte Varianz')
axes[1].grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
plt.show(figure)

### Übung — Projektion auf 2 Dimensionen

Reduzieren Sie die 64 Dimensionen auf **2**, um die Daten zeichnen zu können.

1. Erzeugen Sie ein `PCA(n_components=2)`-Objekt namens `pca_2d`.
2. Wenden Sie `fit_transform` auf `x_digits` an und speichern Sie das Ergebnis in `x_2d`.

In [ ]:
pca_2d = PCA(n_components=2)
x_2d = pca_2d.fit_transform(x_digits)

print(f'Form von x_2d: {x_2d.shape}')
total_explained = pca_2d.explained_variance_ratio_.sum() * 100
print(f'Diese Komponenten behalten zusammen {round(total_explained, 1)}% der Varianz.')

Jetzt können wir die 2D-Projektion als Streudiagramm visualisieren.

In [ ]:
def plot_digits_representation(x: np.ndarray, y: np.ndarray, dimensions: list[str]) -> plt.Figure:
    """
    Erstellt eine zweidimensionale Darstellung des NIST-Datensatzes.
    :param x: Niederdimensionale Repräsentation der Bilder als Array der Form `(n, 2)`.
    :param y: Indizes der Labels aller Beobachtungen Ganzzahl-Array der Form `(n)`.
    :param dimensions: Namen der beiden Dimensionen von `x`. 
    """
    figure, ax = plt.subplots(figsize=(8, 6), dpi=100)
    scatter = ax.scatter(x[:, 0], x[:, 1], c=y, cmap='tab10', s=12, alpha=0.7)
    ax.set(xlabel=dimensions[0], ylabel=dimensions[1])
    digit_labels = list(range(10))
    color_bar = figure.colorbar(scatter, ax=ax, ticks=np.linspace(0.45, 8.55, 10), label='Ziffer')
    color_bar.ax.set_yticklabels(digit_labels)
    return figure

figure = plot_digits_representation(x_2d, y_digits, ['Hauptkomponente 1', 'Hauptkomponente 2'])
plt.show(figure)
del figure

Einige Ziffern (z. B. 0 und 4) trennen sich gut, andere überlappen stark — kein Wunder, denn diese 2 Komponenten behalten nur rund **28 %** der Varianz. PCA ist eben **linear**: Sie darf die Daten nur drehen und strecken, nicht "entfalten". Diese Grenze schauen wir uns in Teil 3 genauer an.

### Rekonstruktion — was bedeutet "erklärte Varianz" anschaulich?

Eine schöne Eigenschaft der PCA ist, dass wir aus der reduzierten Darstellung wieder zurück in den ursprünglichen 64-dimensionalen Raum projizieren können:
$$\hat{\mathbf{X}} = \mathbf{Z}\,\mathbf{V}_k^{\top} + \bar{\mathbf{x}}$$

`PCA`-Objekte in sklearn bieten dafür die Methode [`inverse_transform`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html#sklearn.decomposition.PCA.inverse_transform) an.

Je mehr Komponenten $k$ wir behalten, desto näher liegt die Rekonstruktion $\hat{\mathbf{X}}$ am Original. Bei Bildern können wir das direkt sehen.

### Übung: Rekonstruktion mit k Komponenten

Vervollständigen Sie die Funktion `reconstruct_with_k`. Sie sollte:

1. eine `PCA` mit `n_components=k` an `x` anpassen und `x` transformieren (`fit_transform`),
2. die reduzierten Daten mit `inverse_transform` wieder in den Originalraum zurückprojizieren,
3. das rekonstruierte Array zurückgeben.

In [ ]:
def reconstruct_with_k(x: np.ndarray, k: int) -> np.ndarray:
    """
    Führt eine PCA auf der gegebenen Punktwolke durch und gibt deren rekonstruierte
    Darstellung im ursprünglichen Merkmalsraum zurück.
    :param x: Hochdimensionaler Datensatz als Array der Form `(Beobachtungen, Dimensionen)`.
    :param k: Anzahl der zu verwendenden Hauptkomponenten.
    :return: die Rekonstruktion des gegebenen Datensatzes als Array der Form `x.shape`. 
    """
    pca = PCA(n_components=k)
    x_transformed = pca.fit_transform(x)
    return pca.inverse_transform(x_transformed)


# Ein Beispielbild mit zunehmender Komponentenzahl rekonstruieren und vergleichen
i = 2301
ks = [1, 5, 10, 20, 40]

figure, axes = plt.subplots(1, len(ks) + 1, figsize=(2 * (len(ks) + 1), 2.6), dpi=100)
axes[0].imshow(x_digits[i, ].reshape(8, 8), cmap='gray_r')
axes[0].set_title(str(y_digits[i]))
axes[0].axis('off')
for ax, k in zip(axes[1:], ks):
    x_reconstructed = reconstruct_with_k(x_digits, k)
    ax.imshow(x_reconstructed[i, :].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'K = {k}')
    ax.axis('off')
plt.show()

Mit nur einer Komponente ist kaum etwas zu erkennen; ab etwa $k=20$ ist die Ziffer klar lesbar — und passend dazu hatten wir berechnet, dass rund 21 Komponenten für 90 % der Varianz genügen.

**Erklärte Varianz** wird hier sichtbar: Sie misst, wie gut sich das Original aus wenigen Komponenten wiederherstellen lässt.

### Vor- und Nachteile von PCA

| Vorteile | Nachteile |
|---|---|
| Schnell, deterministisch (kein Zufall) | Nur **lineare** Zusammenhänge — kann nichtlineare Mannigfaltigkeiten nicht "entfalten" |
| Achsen mathematisch interpretierbar (Richtungen maximaler Varianz) | Empfindlich gegenüber Skalierung und Ausreißern |
| Rücktransformation möglich (`inverse_transform`) → Kompression, Entrauschen | Annahme "große Varianz = wichtige Information" stimmt nicht immer |
| Funktioniert auf neuen Datenpunkten (`transform`) | Komponenten oft nicht inhaltlich/semantisch deutbar |

PCA ist deshalb ein hervorragendes Vorverarbeitungs- und Kompressionswerkzeug — aber zur reinen Visualisierung von Clustern ist sie nicht immer ideal. Dafür gibt es nichtlineare Verfahren.

## Teil 3 — Die Grenze linearer Verfahren und ein Blick auf t-SNE

Eine lineare Projektion kann Teile des Punktenwolkes (der Figur) zum Überlappen bringen, die im Hochdimensionalen Raum weit auseinander liegen. Manche Strukturen lassen sich durch bloßes Drehen und Stauchen einfach nicht in der Ebene trennen — man müsste sie **entfalten**. Das können lineare Verfahren wie PCA nicht.

**t-SNE** (t-distributed Stochastic Neighbor Embedding) ist ein nichtlineares Verfahren. Es versucht nicht, Varianz zu maximieren, sondern **lokale Nachbarschaften** zu erhalten: Punkte, die im hochdimensionalen Raum nahe beieinander liegen, sollen auch nach der Transformation nahe beieinander landen.

Die Distanzmetrik ist:

$P_{j|i} = \frac{exp(-d(x_i, x_j)^2) / (2 \sigma_i^2)}{\sum_{k \ne j}{exp(-d(x_i, x_k)^2) / (2 \sigma_i^2)}}$

Laurens van der Maaten (der Author des Algorithmus) erklärt in einem [Vortrag bei Google](https://www.youtube.com/watch?v=RJVL80Gg3lA) wie t-SNE funktioniert.

> Wichtige Warnhinweise — t-SNE-Plots werden oft überinterpretiert:
> - t-SNE ist **stochastisch**: verschiedene Zufalls-Seeds liefern verschiedene Layouts.
> - Abstände zwischen den Clustern und ihre Größen sind nicht aussagekräftig.
> - Es gibt keine sinnvolle Projektion neuer Punkte (kein `transform` wie bei PCA).
> - Der Parameter `perplexity` (grob: angenommene Nachbaranzahl) verändert das Bild stark.
> - t-SNE eignet sich zur Exploration odeer Visualisierung, aber nicht als Vorverarbeitung für ein Modell.

Die Klasse [`TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) in sklearn implementiert dieses Verfahren.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, init='pca',
            learning_rate='auto', random_state=RANDOM_STATE)
x_tsne = tsne.fit_transform(x_digits)

print(f'Form der t-SNE-Einbettung: {x_tsne.shape}')

In [ ]:
figure = plot_digits_representation(x_tsne, y_digits, ['t-SNE 1', 't-SNE 2'])
plt.show(figure)

Vergleichen Sie dieses Bild mit der PCA-2D-Projektion aus Teil 2: t-SNE trennt die zehn Ziffern deutlich sauberer in Cluster. Das ist die Stärke eines nichtlinearen Verfahrens — erkauft mit Stochastik und schlechterer Interpretierbarkeit der Achsen.

Die nachstehenden Diagramme veranschaulichen die Stochastizität dieser Methode. zweimal t-SNE mit zufaelliger Initialisierung und
unterschiedlichem Seed. Die Layouts unterscheiden sich, obwohl die Cluster gleich bleiben.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 4.6), dpi=100)
for ax, seed in zip(axes, [27, 26]):
    emb = TSNE(n_components=2, perplexity=30, init='random',
               learning_rate='auto', random_state=seed).fit_transform(x_digits)
    ax.scatter(emb[:, 0], emb[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
    ax.set_title(f'random_state = {seed}'); ax.set_xticks([]); ax.set_yticks([])
plt.show(figure)

### Mit den Hyperparametern spielen

Wie schon bei den Entscheidungsbäumen (Baumtiefe, Unreinheitsmaß, ...) lohnt sich auch hier das
Experimentieren. Gehen Sie zu **Aufgabe 5** zurück und probieren Sie verschiedene Werte für `perplexity`
(z. B. 5, 30, 50) sowie unterschiedliche `random_state`-Werte aus. Diskutieren Sie:

- Wie verändert kleine vs. große `perplexity` das Bild?
- Bleiben die Cluster bei verschiedenen Seeds dieselben, auch wenn ihre Lage wechselt?
- Was bedeutet das für die Interpretation eines einzelnen t-SNE-Plots?

### PCA vs. t-SNE — Überblick

| | **PCA** | **t-SNE** |
|---|---|---|
| Art | linear | nichtlinear |
| Erhält | globale Varianz / Struktur | lokale Nachbarschaften |
| Deterministisch? | ja | nein (zufallsabhängig) |
| Neue Punkte projizierbar? | ja (`transform`) | nein |
| Achsen interpretierbar? | ja | nein |
| Geschwindigkeit | schnell | langsam |
| Typischer Einsatz | Vorverarbeitung, Kompression, Entrauschen | Exploration, Cluster-Visualisierung |

## Zusammenfassung

- **Dimensionsreduktion** projiziert hochdimensionale Daten auf wenige Dimensionen — wie der Schatten unserer Origami-Figur.
- **PCA** sucht orthogonale Richtungen **maximaler Varianz** über die Eigenzerlegung der Kovarianzmatrix.
- **Standardisierung** ist nötig, wenn Merkmale verschiedene Skalen haben (Penguins), aber unnötig bzw. schädlich bei gleichskalierten Merkmalen (Ziffern-Pixel).
- **Erklärte Varianz** und **Rekonstruktion** zeigen, wie viel Information wenige Komponenten behalten.
- PCA ist **linear** und stößt bei nichtlinearen Strukturen an Grenzen; **t-SNE** entfaltet solche Strukturen, ist aber stochastisch und nur zur Visualisierung gedacht.

## Hausaufgabe — MDS und UMAP

Zwei weitere Verfahren zum selbstständigen Ausprobieren.

**1. MDS** (Multidimensional Scaling, [Multidimensionale Skalierung](https://de.wikipedia.org/wiki/Multidimensionale_Skalierung))  versucht, paarweise Abstände zu erhalten,
```python
from sklearn.manifold import MDS
# Hinweis: MDS ist langsam, daher nur auf einer Teilstichprobe rechnen
subset = rng.choice(len(x_digits), size=500, replace=False)
x_mds = MDS(n_components=2, random_state=RANDOM_STATE).fit_transform(x_digits[subset])
# x_mds nach y_digits[subset] eingefärbt zeichnen
```
*Aufgabe:* Erhält MDS eher die globale oder die lokale Struktur? Vergleichen Sie mit t-SNE.

**2. UMAP** ([Uniform Manifold Approximation and Projection](https://umap-learn.readthedocs.io/en/latest/), benötigt das zusätzliche Paket `umap-learn`).
```bash
pip install umap-learn
```
```python
import umap  # nach der Installation
reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE)
x_umap = reducer.fit_transform(x_digits)
# x_umap wie zuvor nach y_digits eingefärbt zeichnen
```
*Aufgabe:* Vergleichen Sie das UMAP-Ergebnis mit PCA und t-SNE. Wie schnell ist UMAP? Wie verändert der Parameter `n_neighbors` das Bild?